In [0]:
CREATE OR REPLACE VIEW airline_catalog.semantic.vw_destination_airport_performance AS -- Crear o reemplazar vista de desempeño de aeropuertos destino

WITH airport_metrics AS -- CTE para métricas de aeropuerto
(
    SELECT

        da.airport_id, -- ID del aeropuerto
        da.airport_code, -- Código del aeropuerto
        da.airport_name, -- Nombre del aeropuerto

        CONCAT_WS(', ', da.city, da.state) AS location, -- Ubicación (Ciudad, Estado)

        COUNT(*) AS total_arrivals, -- Total de llegadas

        ROUND(AVG(ff.arrival_delay), 2) AS avg_arrival_delay, -- Promedio de retraso en llegada

        SUM(CASE
                WHEN ff.flight_status = 'Delayed' THEN 1 -- Cuenta vuelos retrasados
                ELSE 0
            END) AS delayed_arrivals, -- Total de llegadas retrasadas

        ROUND(
            100.0 *
            SUM(CASE WHEN ff.flight_status = 'Delayed' THEN 1 ELSE 0 END) -- Porcentaje de llegadas retrasadas
            / COUNT(*),
            2
        ) AS pct_delayed, -- Porcentaje de retrasos

        SUM(CASE
                WHEN ff.cancelled THEN 1 -- Cuenta vuelos cancelados
                ELSE 0
            END) AS cancelled_flights, -- Total de vuelos cancelados

        ROUND(
            100.0 *
            SUM(CASE WHEN ff.cancelled THEN 1 ELSE 0 END) -- Porcentaje de vuelos cancelados
            / COUNT(*),
            2
        ) AS pct_cancelled, -- Porcentaje de cancelaciones
        ROUND(AVG(ff.distance),2) AS avg_distance, -- Promedio de distancia recorrida
        SUM(ff.distance) AS total_distance -- Total de distancia recorrida


    FROM airline_catalog.gold.fact_flights ff -- Fuente: tabla de hechos de vuelos

    INNER JOIN airline_catalog.gold.dim_airport da -- Unión con dimensión aeropuerto
        ON ff.destination_airport_id = da.airport_id -- Condición de unión por aeropuerto destino

    GROUP BY

        da.airport_id, -- Agrupa por ID de aeropuerto
        da.airport_code, -- Agrupa por código de aeropuerto
        da.airport_name, -- Agrupa por nombre de aeropuerto
        da.city, -- Agrupa por ciudad
        da.state -- Agrupa por estado
)

SELECT

    *, -- Selecciona todas las columnas de métricas

    RANK() OVER (
        ORDER BY total_arrivals DESC -- Ranking por volumen de llegadas (mayor a menor)
    ) AS traffic_rank, -- Ranking de tráfico

    RANK() OVER (
        ORDER BY avg_arrival_delay ASC -- Ranking por puntualidad (menor retraso primero)
    ) AS punctuality_rank, -- Ranking de puntualidad

    DENSE_RANK() OVER (
        ORDER BY pct_delayed ASC -- Ranking por menor porcentaje de retrasos
    ) AS delay_rank -- Ranking de retrasos

FROM airport_metrics; -- Fuente: CTE de métricas de aeropuerto

In [0]:
select * from airline_catalog.semantic.vw_destination_airport_performance